## 1. Instalação

O **PuLP** é a biblioteca que vamos usar para modelar e resolver problemas de Programação Linear e Inteira.

Ele já vem com um solver embutido (CBC), então não precisa instalar mais nada.

In [ ]:
!pip install pulp -q

In [ ]:
from pulp import *

---
## 2. Problema do Seu João (exemplo guiado)

Seu João tem uma padaria e produz **pães** e **bolos**.

| Produto | Lucro | Farinha | Tempo |
|---------|-------|---------|-------|
| Pão     | R$ 2  | 1 kg    | 1 h   |
| Bolo    | R$ 5  | 2 kg    | 1 h   |
| **Disponível** | - | **10 kg** | **6 h** |

**Objetivo:** Maximizar o lucro.

### Passo 1 — Criar o problema

In [ ]:
# Criamos o problema e dizemos que queremos MAXIMIZAR
prob = LpProblem("Padaria_Seu_Joao", LpMaximize)

### Passo 2 — Definir as variáveis de decisão

- x1 = quantidade de pães
- x2 = quantidade de bolos
- `lowBound=0` garante que x ≥ 0 (não-negatividade)

In [ ]:
x1 = LpVariable("paes", lowBound=0)
x2 = LpVariable("bolos", lowBound=0)

### Passo 3 — Função objetivo

Max Z = 2x₁ + 5x₂

No PuLP, o primeiro `+=` é sempre a função objetivo.

In [ ]:
prob += 2*x1 + 5*x2, "Lucro"

### Passo 4 — Restrições

- Farinha: x₁ + 2x₂ ≤ 10
- Tempo: x₁ + x₂ ≤ 6

In [ ]:
prob += x1 + 2*x2 <= 10, "Farinha"
prob += x1 +   x2 <= 6,  "Tempo"

### Passo 5 — Resolver!

In [ ]:
prob.solve(PULP_CBC_CMD(msg=0))

print("=" * 35)
print("  RESULTADO — Padaria do Seu João")
print("=" * 35)
print(f"  Status: {LpStatus[prob.status]}")
print(f"  Pães:   {x1.varValue:.0f}")
print(f"  Bolos:  {x2.varValue:.0f}")
print(f"  Lucro:  R$ {value(prob.objective):.2f}")
print("=" * 35)

### Análise das folgas

Quais recursos sobraram? Quais foram usados por completo?

In [ ]:
print("\nAnálise das restrições:")
print("-" * 35)
for name, constraint in prob.constraints.items():
    folga = constraint.slack
    status = "✅ Esgotado" if folga == 0 else f"Sobrou {folga}"
    print(f"  {name}: {status}")

Reparem: a **farinha esgotou** (folga = 0) e o **tempo sobrou** 1 hora. Isso bate com o que a gente viu no método gráfico e no Simplex.

---

## 3. Exercício — Fábrica de Móveis 🪑

Uma fábrica produz **cadeiras** e **mesas**.

| Produto | Lucro | Madeira (m²) | Mão de obra (h) |
|---------|-------|-------------|------------------|
| Cadeira | R$ 40 | 2           | 3                |
| Mesa    | R$ 70 | 5           | 4                |
| **Disponível** | - | **30 m²** | **28 h** |

**Objetivo:** Maximizar o lucro.

Usem o código do Seu João como base — só mudem os números!

In [ ]:
# EXERCÍCIO: completem o código abaixo

prob2 = LpProblem("Fabrica_Moveis", LpMaximize)

# Variáveis de decisão
x1 = LpVariable("cadeiras", lowBound=0)
x2 = LpVariable("mesas", lowBound=0)

# Função objetivo
# prob2 += ...  # <-- preencham aqui

# Restrições
# prob2 += ...  # madeira
# prob2 += ...  # mão de obra

# Resolver
prob2.solve(PULP_CBC_CMD(msg=0))

print(f"Cadeiras: {x1.varValue}")
print(f"Mesas: {x2.varValue}")
print(f"Lucro: R$ {value(prob2.objective)}")

### Solução

In [ ]:
prob2 = LpProblem("Fabrica_Moveis", LpMaximize)

x1 = LpVariable("cadeiras", lowBound=0)
x2 = LpVariable("mesas", lowBound=0)

# Função objetivo: maximizar lucro
prob2 += 40*x1 + 70*x2, "Lucro"

# Restrições
prob2 += 2*x1 + 5*x2 <= 30, "Madeira"
prob2 += 3*x1 + 4*x2 <= 28, "Mao_de_obra"

prob2.solve(PULP_CBC_CMD(msg=0))

print("=" * 35)
print("  RESULTADO — Fábrica de Móveis")
print("=" * 35)
print(f"  Status:   {LpStatus[prob2.status]}")
print(f"  Cadeiras: {x1.varValue:.4f}")
print(f"  Mesas:    {x2.varValue:.4f}")
print(f"  Lucro:    R$ {value(prob2.objective):.2f}")
print("=" * 35)

print(f"\n⚠️  Cadeiras = {x1.varValue:.4f} → Não é inteiro!")
print(f"⚠️  Mesas = {x2.varValue:.4f} → Não é inteiro!")
print(f"\nComo fabricar pedaço de móvel? 🤔")
print(f"Arredondar pra baixo? Pra cima? Será que é a melhor decisão?")
print(f"\n→ Amanhã: Programação INTEIRA resolve isso.")

### E se a gente arredondar?

Vamos ver o que acontece se tentarmos arredondar pra baixo (2 cadeiras, 4 mesas):

In [ ]:
import math

# Solução ótima (fracionária)
c_otimo = x1.varValue
m_otimo = x2.varValue
z_otimo = value(prob2.objective)

# Arredondar pra baixo
c_floor = math.floor(c_otimo)
m_floor = math.floor(m_otimo)
z_floor = 40*c_floor + 70*m_floor

# Verificar viabilidade
madeira_usada = 2*c_floor + 5*m_floor
mao_usada = 3*c_floor + 4*m_floor

print("Comparação:")
print(f"{'':>20} {'PL (fracionário)':>18} {'Arredondado ↓':>15}")
print("-" * 55)
print(f"{'Cadeiras':>20} {c_otimo:>18.4f} {c_floor:>15}")
print(f"{'Mesas':>20} {m_otimo:>18.4f} {m_floor:>15}")
print(f"{'Lucro':>20} {'R$ ' + f'{z_otimo:.2f}':>18} {'R$ ' + f'{z_floor:.2f}':>15}")
print(f"{'Madeira':>20} {'30.00':>18} {madeira_usada:>15}")
print(f"{'Mão de obra':>20} {'28.00':>18} {mao_usada:>15}")

print(f"\n💸 Perda por arredondar: R$ {z_otimo - z_floor:.2f}")
print(f"\nE se tentasse (3, 4)? Lucro = R$ {40*3 + 70*4:.2f}")
print(f"   Madeira: {2*3 + 5*4} / 30 ✅  |  Mão de obra: {3*3 + 4*4} / 28 ✅")
print(f"\n→ Arredondar NEM SEMPRE dá a melhor solução inteira!")

---

## 4. Exercício — Planos de IA para a Verboo 🤖 (caso real)

A Verboo tem **6 pessoas** na equipe. Cada uma precisa de uma ferramenta de IA.

| Plano | Custo/mês (R$) | Produtividade (1-5) | Capacidade (1-10) |
|---|---|---|---|
| Codex Go | 45 | 3 | 2 |
| Codex Plus | 110 | 4 | 5 |
| Claude Pro | 110 | 4 | 7 |
| Claude Max | 550 | 5 | 10 |
| Gemini AI Plus | 25 | 3 | 3 |
| Gemini AI Pro | 97 | 4 | 8 |

**Budget:** R$ 2.000/mês

**Restrições:**
- Todas as 6 pessoas precisam de um plano (==)
- Capacidade total do time ≥ 30

**Objetivo:** Maximizar a produtividade total do time.

In [ ]:
# EXERCÍCIO: completem o código abaixo

prob3 = LpProblem("Planos_IA", LpMaximize)

x1 = LpVariable("Codex_Go", lowBound=0)
x2 = LpVariable("Codex_Plus", lowBound=0)
x3 = LpVariable("Claude_Pro", lowBound=0)
x4 = LpVariable("Claude_Max", lowBound=0)
x5 = LpVariable("Gemini_Plus", lowBound=0)
x6 = LpVariable("Gemini_Pro", lowBound=0)

# Função objetivo: maximizar produtividade
# prob3 += ...  # <-- preencham aqui

# Restrições
# prob3 += ...  # budget (R$ 2000)
# prob3 += ...  # 6 pessoas (atenção: aqui é ==, não <=)
# prob3 += ...  # capacidade mínima >= 30

prob3.solve(PULP_CBC_CMD(msg=0))

print(f"Codex Go:       {x1.varValue:.4f} licenças")
print(f"Codex Plus:     {x2.varValue:.4f} licenças")
print(f"Claude Pro:     {x3.varValue:.4f} licenças")
print(f"Claude Max:     {x4.varValue:.4f} licenças")
print(f"Gemini Plus:    {x5.varValue:.4f} licenças")
print(f"Gemini Pro:     {x6.varValue:.4f} licenças")
print(f"Produtividade:  {value(prob3.objective):.2f}")

### Solução — Planos de IA

In [ ]:
prob3 = LpProblem("Planos_IA", LpMaximize)

x1 = LpVariable("Codex_Go", lowBound=0)
x2 = LpVariable("Codex_Plus", lowBound=0)
x3 = LpVariable("Claude_Pro", lowBound=0)
x4 = LpVariable("Claude_Max", lowBound=0)
x5 = LpVariable("Gemini_Plus", lowBound=0)
x6 = LpVariable("Gemini_Pro", lowBound=0)

prob3 += 3*x1 + 4*x2 + 4*x3 + 5*x4 + 3*x5 + 4*x6, "Produtividade"
prob3 += 45*x1 + 110*x2 + 110*x3 + 550*x4 + 25*x5 + 97*x6 <= 2000, "Budget"
prob3 += x1 + x2 + x3 + x4 + x5 + x6 == 6, "Pessoas"
prob3 += 2*x1 + 5*x2 + 7*x3 + 10*x4 + 3*x5 + 8*x6 >= 30, "Capacidade_minima"

prob3.solve(PULP_CBC_CMD(msg=0))

print("=" * 50)
print("  RESULTADO — Planos de IA (Verboo)")
print("=" * 50)
for v in prob3.variables():
    if v.varValue > 0.001:
        print(f"  {v.name}: {v.varValue:.4f} licenças")
custo = 45*x1.varValue+110*x2.varValue+110*x3.varValue+550*x4.varValue+25*x5.varValue+97*x6.varValue
cap = 2*x1.varValue+5*x2.varValue+7*x3.varValue+10*x4.varValue+3*x5.varValue+8*x6.varValue
print(f"\n  Produtividade: {value(prob3.objective):.2f}")
print(f"  Custo: R$ {custo:.0f} / 2000")
print(f"  Capacidade: {cap:.1f}")
print("=" * 50)

print(f"\n⚠️  Resultado fracionário! Não existe 0.13 de uma licença.")
print(f"→ Amanhã: Programação INTEIRA resolve isso!")

---

## Resumo

| Etapa | Modelo na mão | Código PuLP |
|-------|--------------|-------------|
| Criar problema | — | `LpProblem("nome", LpMaximize)` |
| Variáveis | x₁, x₂ ≥ 0 | `LpVariable("x", lowBound=0)` |
| Objetivo | Max 2x₁ + 5x₂ | `prob += 2*x1 + 5*x2` |
| Restrições (≤) | x₁ + 2x₂ ≤ 10 | `prob += x1 + 2*x2 <= 10` |
| Restrições (≥) | 2x₁ + 3x₂ ≥ 30 | `prob += 2*x1 + 3*x2 >= 30` |
| Igualdade | x₁ + x₂ = 6 | `prob += x1 + x2 == 6` |
| Resolver | Simplex / Gráfico | `prob.solve()` |
| Ler resultado | Ler no tableau | `x1.varValue`, `value(prob.objective)` |
| Folgas | Verificar recursos | `prob.constraints["nome"].slack` |

**O que fizemos hoje:**
1. Seu João → PL básica, resultado inteiro
2. Fábrica de Móveis → resultado **fracionário** (não faz sentido!)
3. Planos de IA (Verboo) → 6 variáveis, 3 tipos de restrição, resultado fracionário

**Amanhã:** Programação Inteira — quando o resultado precisa ser inteiro!

**Exercícios para casa:** abram o notebook de exercícios!